---
## 7. Shannon entropy — measuring conservation

We want a single number that captures how "conserved" an alignment
position is. Before we see the formula, let's think about what
such a number should look like.

### 7.1 What should "conservation" mean?

Look at these three alignment columns (4 species each):

| Column | Species 1 | Species 2 | Species 3 | Species 4 |
|--------|-----------|-----------|-----------|-----------|
| A | Ala | Ala | Ala | Ala |
| B | Ala | Gly | Ala | Gly |
| C | Ala | Gly | Leu | Lys |

### Exercise 6a

1. **Rank** columns A, B, C from most conserved to least conserved.
2. If you had to assign a **single number** (say, between 0 and 5)
   to each column where 0 = perfectly conserved and higher = more
   variable, what values would you give?
3. What property are you measuring? How would you describe it in
   words?

In [ ]:
# Think about it before scrolling down!

### 7.2 Shannon entropy

**Shannon entropy** does exactly this — it measures how
"unpredictable" a position is.

$$H = -\sum_{i=1}^{k} p_i \cdot \log_2(p_i)$$

where $p_i$ is the frequency of amino acid $i$, and $k$ is the
number of distinct amino acids.

- Perfectly conserved (one amino acid): $H = 0$
- Two amino acids at 50/50: $H = 1.0$
- All 20 amino acids equally likely: $H = \log_2(20) \approx 4.3$

Low entropy = conserved. High entropy = variable.

### 7.3 Computing entropy step by step

In [ ]:
# Let's compute entropy at position 100 by hand
col = list(aln[100])

# Remove gaps
residues = [aa for aa in col if aa != '-']
print(f"Position 100: {len(residues)} residues (no gaps)")

# Count each amino acid
counts = Counter(residues)
total = len(residues)
print(f"Counts: {dict(counts)}")
print()

# Compute frequencies and entropy contributions
entropy = 0.0
for aa, count in counts.items():
    p = count / total
    contribution = -p * math.log2(p)
    entropy += contribution
    print(f"  {aa}: p = {count}/{total} = {p:.3f}"
          f"  →  -p·log₂(p) = {contribution:.4f}")

print(f"\nShannon entropy at position 100: {entropy:.3f}")

### ✏️ Exercise 6b

Now **write a function** that computes Shannon entropy for a list
of amino acids (ignoring gaps):

```python
def shannon_entropy(column):
    """Shannon entropy of a list of amino acids (ignoring gaps)."""
    # 1. Remove gaps
    # 2. Count each amino acid
    # 3. Compute frequencies
    # 4. Apply the formula: H = -Σ p·log₂(p)
    # 5. Return H
    pass
```

Test your function:
- All same amino acid → should return 0.0
- Half A, half G → should return 1.0
- Position 100 → should match the calculation above

In [ ]:
def shannon_entropy(column):
    """Shannon entropy of a list of amino acids (ignoring gaps)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Test it
# print(shannon_entropy(['A', 'A', 'A', 'A']))          # → 0.0
# print(shannon_entropy(['A', 'G', 'A', 'G']))          # → 1.0
# print(shannon_entropy(list(aln[100])))                 # → should match above

### 7.4 Conservation profile across the whole alignment

In [ ]:
n_pos = aln.shape[1]
entropies = [shannon_entropy(list(aln[pos])) for pos in range(n_pos)]

print(f"Computed entropy for {n_pos} positions")
print(f"  Min: {min(entropies):.3f}  (most conserved)")
print(f"  Max: {max(entropies):.3f}  (most variable)")
print(f"  Mean: {np.mean(entropies):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(range(n_pos), entropies, alpha=0.4, color='steelblue')
ax.plot(range(n_pos), entropies, linewidth=0.5, color='steelblue')
ax.set_xlabel('Alignment position')
ax.set_ylabel('Shannon entropy')
ax.set_title('Prestin (SLC26A5) — conservation profile')
ax.set_xlim(0, n_pos)
plt.tight_layout()
plt.show()

Valleys = highly conserved regions. Peaks = variable regions.

### Exercise 7

1. How many positions have entropy = 0 (perfectly conserved)?
   What fraction of the alignment is that?

2. What are the 10 **most conserved** positions with non-zero entropy?
   What amino acids are at those positions?

3. What are the 10 **most variable** positions? Are they near the
   beginning/end of the alignment (where trimming may be imperfect)?

4. Compute the **mean entropy** separately for echolocators and
   non-echolocators. Is the prestin protein more or less conserved
   in echolocators?

In [ ]:
# Your code here

---
## 8. Prestin 3D structure

Human prestin has a cryo-EM structure: **PDB 7S8X** (Bavi et al.
2021). We can visualize it inline using `py3Dmol`.

The protein has three main regions:
- **TMD** (transmembrane domain, ~residues 81–505) — the motor domain
- **Linker** (~506–530)
- **STAS domain** (~531–744) — cytoplasmic regulatory domain

See also: https://www.uniprot.org/uniprotkb/P58743/entry#structure

In [ ]:
try:
    import py3Dmol

    view = py3Dmol.view(query='pdb:7S8X', width=700, height=500)

    # Remove chain B (show monomer only)
    view.removeModel(1)

    # Base: transparent cartoon
    view.setStyle({'chain': 'A'},
                  {'cartoon': {'color': 'gray', 'opacity': 0.5}})

    # TMD in red
    view.addStyle({'chain': 'A', 'resi': list(range(81, 506))},
                  {'cartoon': {'color': 'red', 'opacity': 0.9}})

    # Linker in orange
    view.addStyle({'chain': 'A', 'resi': list(range(506, 531))},
                  {'cartoon': {'color': 'orange', 'opacity': 0.9}})

    # STAS in blue
    view.addStyle({'chain': 'A', 'resi': list(range(531, 745))},
                  {'cartoon': {'color': 'blue', 'opacity': 0.9}})

    view.zoomTo({'chain': 'A'})
    view.show()

except ImportError:
    print("py3Dmol not installed — run: pip install py3Dmol")
    print("Explore the structure at: https://www.rcsb.org/3d-view/7S8X")

The TMD (red) is where prestin's motor function resides. If
convergent evolution targets echolocation-related function, we might
expect convergent sites to concentrate there. We'll return to this
idea in the assignment.